<a href="https://colab.research.google.com/github/veniola/dx704-project-11/blob/main/dx704-project-11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DX 704 Week 11 Project

In this project, you will develop and test prompts asking a language model to classify text from a home services query and match it to an appropriate category of home services.

The full project description and a template notebook are available on GitHub: [Project 11 Materials](https://github.com/bu-cds-dx704/dx704-project-11).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1 : Design a Short Prompt

The provided file "queries.txt" contains sample text from requests by homeowners by email or phone.
These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services.
The provided file has columns query_id, query, and target_category.
Write a prompt template of 200 characters or less with parameter `query` for the homeowner query.
Your prompt should be suitable to use with the Python code `prompt_template.format(query=query)`.
Test your prompt with the model `gpt-6-luna` and suitable parsing code.

In [1]:
# =========================
# -- Important Libraries --
# =========================

import pandas as pd
import io
import os

In [2]:
%pip install -U "openai>=3.26.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 12.7 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


In [3]:
from google.colab import userdata
from openai import OpenAI

In [4]:
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

In [5]:
response = client.responses.create(
    model="gpt-6-luna",
    input="How much wood would a woodchuck chuck if a wood chuck could chuck wood?"
)

print(response.output_text)

As much wood as a woodchuck could chuck—if a woodchuck could chuck wood. The often-quoted estimate is **about 700 pounds**, but that’s a playful guess, not a real woodchucking measurement.


In [10]:
# ===================================
# -- Part 1: Design A Short Prompt --
# ===================================

# Load queries
queries = pd.read_csv('queries.txt', sep='\t')
display_queries = queries.head()
display_queries

# Prompt template
prompt_template = 'Classify "{query}" into one word: electrical, plumbing, or roofing.'

# Save prompt template to txt file
with open('short-prompt.txt', 'w') as f:
    f.write(prompt_template)

# Test prompt on all queries
results = []

for idx, row in queries.iterrows():
    formatted_prompt = prompt_template.format(query=row['query'])
    response = client.responses.create(
        model="gpt-6-luna",
        input=formatted_prompt
    )

    prediction = response.output_text.strip().lower().replace('.', '')
    results.append({
        'query_id': row['query_id'],
        'predicted_category': prediction
    })

Save your prompt template in a file "short-prompt.txt".
Save the results of your prompt testing in "short-output.tsv" with columns `query_id` and `predicted_category`.

In [13]:
# ===============================================================
# -- Save Prompt Template to TSV File Named 'short-output.tsv' --
# ===============================================================

short_output_df = pd.DataFrame(results)
print(short_output_df)

# Save file
short_output_df.to_csv('short-output.tsv', sep='\t', index=False)
short_output_df.head()

    query_id predicted_category
0          1            roofing
1          2           plumbing
2          3         electrical
3          4            roofing
4          5           plumbing
5          6         electrical
6          7            roofing
7          8           plumbing
8          9         electrical
9         10            roofing
10        11           plumbing
11        12         electrical
12        13            roofing
13        14           plumbing
14        15         electrical
15        16            roofing
16        17           plumbing
17        18         electrical
18        19            roofing
19        20           plumbing
20        21         electrical
21        22            roofing
22        23           plumbing
23        24         electrical
24        25            roofing
25        26           plumbing
26        27         electrical
27        28            roofing
28        29           plumbing
29        30         electrical
30      

,query_id,predicted_category
0,1,roofing
1,2,plumbing
2,3,electrical
3,4,roofing
4,5,plumbing


Submit "short-prompt.txt" and "short-output.tsv" in Gradescope.

Hint: your prompt may be re-tested with the Gemini API, so do not rely solely on lucky language model responses.

## Part 2: Find Short Prompt Mistakes

Construct 5 queries of 100 characters or less that trick your short prompt so that the wrong category is chosen.


In [14]:
# ========================================
# -- Part 2: Find Short Prompt Mistakes --
# ========================================

query_1 = 'There is water inside my hair dryer, how can I fix it?'
query_2 = 'My oven smells like sewage.'
query_3 = 'I need a solar panel mounted directly onto my roof.'
query_4 = 'There is water is leaking near the wall outlet.'
query_5 = 'How to hook up the water filtration system in my new refridgerator?'

trick_queries = [
    {"query": query_1, "target_category": "electrical"},
    {"query": query_2, "target_category": "electrical"},
    {"query": query_3, "target_category": "electrical"},
    {"query": query_4, "target_category": "electrical"},
    {"query": query_5, "target_category": "plumbing"}
]

Save your 5 queries in a file "mistakes.tsv" with columns `query`, `target_category` and `predicted_category`.

In [15]:
# =====================================================
# -- Save 5 Queries to TSV File Named 'mistakes.tsv' --
# =====================================================

mistake_results = []

for item in trick_queries:
    formatted_prompt = prompt_template.format(query=item['query'])
    response = client.responses.create(
        model="gpt-6-luna",
        input=formatted_prompt
    )
    prediction = response.output_text.strip().lower().replace('.', '')

    mistake_results.append({
        'query': item['query'],
        'target_category': item['target_category'],
        'predicted_category': prediction
    })

mistakes_df = pd.DataFrame(mistake_results)
print(mistakes_df)

# Save file
mistakes_df.to_csv('mistakes.tsv', sep='\t', index=False)
mistakes_df.head()

                                               query target_category  \
0  There is water inside my hair dryer, how can I...      electrical   
1                        My oven smells like sewage.      electrical   
2  I need a solar panel mounted directly onto my ...      electrical   
3    There is water is leaking near the wall outlet.      electrical   
4  How to hook up the water filtration system in ...        plumbing   

  predicted_category  
0         electrical  
1           plumbing  
2            roofing  
3           plumbing  
4           plumbing  


,query,target_category,predicted_category
0,"There is water inside my hair dryer, how can I...",electrical,electrical
1,My oven smells like sewage.,electrical,plumbing
2,I need a solar panel mounted directly onto my ...,electrical,roofing
3,There is water is leaking near the wall outlet.,electrical,plumbing
4,How to hook up the water filtration system in ...,plumbing,plumbing


Submit "mistakes.tsv" in Gradescope.

## Part 3: Design a Long Prompt

Repeat part 1 with a length limit of 5000 characters.

In [16]:
# ==================================
# -- Part 3: Design A Long Prompt --
# ==================================

# Repeat part 1 with a length limit of 5000 characters
# Prompt template
prompt_template2 = """
Classify the homeowner query into one word: electrical, plumbing, or roofing.

Rules for resolution:
1. If water is interacting with an electrical device or outlet (e.g. wet hair dryer, wet outlet), classify as "electrical" because an electrician is needed to inspect/fix the appliance/outlet safety.
2. Household appliances with heating elements (ovens, cooktops) are "electrical". If they smell like gas or sewage, they still require appliance/electrical servicing.
3. Solar panels are "electrical" even if mounted to a roof.
4. Refrigerator ice/water hookups are "plumbing".

Examples:
- Query: "There is water inside my hair dryer, how can I fix it?"
- Category: electrical

- Query: "My oven smells like sewage."
- Category: electrical

- Query: "I need a solar panel mounted directly onto my roof."
- Category: electrical

- Query: "There is water is leaking near the wall outlet."
- Category: electrical

- Query: "How to hook up the water filtration system in my new refrigerator?"
- Category: plumbing

- Query: "Need help hooking up the heating coils on my hot water tank."
- Category: plumbing

Query: "{query}"
Answer with exactly one word (electrical, plumbing, or roofing) in lowercase with no punctuation.
"""

# Save prompt template to txt file
with open('long-prompt.txt', 'w') as f:
    f.write(prompt_template2)

# Test prompt on all queries
results = []

for idx, row in queries.iterrows():
    formatted_prompt = prompt_template2.format(query=row['query'])
    response = client.responses.create(
        model="gpt-6-luna",
        input=formatted_prompt
    )

    prediction = response.output_text.strip().lower().replace('.', '')
    results.append({
        'query_id': row['query_id'],
        'predicted_category': prediction
    })


Save your longer prompt template in a file "long-prompt.txt".
Save the results of your prompt testing in "long-output.tsv".
Both files should use the same columns as part 1.

In [17]:
# ====================================================================
# -- Save Long Prompt Template to TSV File Named 'long-prompt.tsv' --
# ====================================================================

long_prompt_df = pd.DataFrame(results)
print(long_prompt_df)

# Save file
long_prompt_df.to_csv('long-prompt.tsv', sep='\t', index=False)
long_prompt_df.head()

    query_id predicted_category
0          1            roofing
1          2           plumbing
2          3         electrical
3          4            roofing
4          5           plumbing
5          6         electrical
6          7            roofing
7          8           plumbing
8          9         electrical
9         10            roofing
10        11           plumbing
11        12         electrical
12        13            roofing
13        14           plumbing
14        15         electrical
15        16            roofing
16        17           plumbing
17        18         electrical
18        19            roofing
19        20           plumbing
20        21         electrical
21        22            roofing
22        23           plumbing
23        24         electrical
24        25            roofing
25        26           plumbing
26        27         electrical
27        28            roofing
28        29           plumbing
29        30         electrical
30      

,query_id,predicted_category
0,1,roofing
1,2,plumbing
2,3,electrical
3,4,roofing
4,5,plumbing


Submit "long-prompt.txt" and "long-output.tsv" in Gradescope.

## Part 4: Use the Decisions API

In September 2026, TypeSafe AI announced [Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev), a new variant of large language models focused on decision making.
In October 2026, OpenAI introduced a similar API that they called a [Decisions API](https://openai.com/index/decisions-api/).
Use the Decisions API ([docs](https://developers.openai.com/api/docs/guides/decisions)) to the implement the same decisions.
Save the results of your testing to "decision-output.tsv" with the same columns as parts 1 and 3.

In [20]:
# ===================================
# -- Part 4: Use the Decisions API --
# ===================================

# Use 'Decisions API' to implement same decisions
import requests
from bs4 import BeautifulSoup

def read_web_link(url):
    # Send a GET request to the URL
    try:
        response = requests.get(url, timeout=10)
        # Check if the request was successful
        if response.status_code == 200:
            # Parse the HTML content
            soup = BeautifulSoup(response.text, 'html.parser')

            # Remove script and style elements
            for script in soup(["script", "style"]):
                script.decompose()

            # Get text and clean up whitespace
            text = soup.get_text()
            lines = (line.strip() for line in text.splitlines())
            chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
            clean_text = '\n'.join(chunk for chunk in chunks if chunk)
            return clean_text
    except Exception as e:
        pass
    return None

# Alternative
scraped_content = read_web_link('https://developers.openai.com/api/docs/guides/decisions$0')

if scraped_content and "Failed" not in scraped_content:
    prompt_template3 = scraped_content + "\n\nQuery: \"{query}\"\nAnswer:"
else:
    prompt_template3 = """

    Your task is to classify a homeowner's query into one of three categories: electrical, plumbing, or roofing.

    We need correct decision logic for tricky cases:
    1. Water inside an electrical device or near an electrical outlet -> electrical
    2. Cooking appliances with heating elements (ovens, cooktops) -> electrical (even with sewage/gas smells)
    3. Solar panels mounted on roof -> electrical
    4. Refrigerator water/ice lines -> plumbing

    Query: "{query}"
    Response (provide only the lowercase category name with no trailing punctuation):
    """

# Save prompt template to txt file
with open('decisions-prompt.txt', 'w') as f:
    f.write(prompt_template3)

# Test prompt on all queries
results = []

for idx, row in queries.iterrows():
    formatted_prompt = prompt_template3.format(query=row['query'])
    response = client.responses.create(
        model="gpt-6-luna",
        input=formatted_prompt
    )

    prediction = response.output_text.strip().lower().replace('.', '')
    results.append({
        'query_id': row['query_id'],
        'predicted_category': prediction
    })

In [21]:
# ===========================================================
# -- Save Results to TSV File Named 'decision-outputs.tsv' --
# ===========================================================

decision_outputs_df = pd.DataFrame(results)
print(decision_outputs_df.head())

# Save file
decision_outputs_df.to_csv('decision-outputs.tsv', sep='\t', index=False)

   query_id predicted_category
0         1            roofing
1         2           plumbing
2         3         electrical
3         4            roofing
4         5           plumbing


## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.